In [2]:
import yfinance as yf
import pandas as pd
from datetime import datetime
from pathlib import Path


### Download tickers using yf

In [3]:
DATA_STORE = Path('..', 'data', 'nasdaq_data_yf.h5')


In [4]:
def get_sp500_tickers():
    sp500 = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')[0]
    sp500['Symbol'] = sp500['Symbol'].str.replace('.', '-')
    symbols_list = sp500['Symbol'].unique().tolist()
    return symbols_list
def load_data(tickers='APPL',end_date='2024-01-01',years=8):
    start_date = pd.to_datetime(end_date)-pd.DateOffset(365*years)
    df = yf.download(tickers, start=start_date, end=end_date, auto_adjust=False)
    #df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    #df = df.dropna()
    return df

def get_nasdaq_tickers():
    nasdaq_tickers = pd.read_json('../data/nasdaq_tickers.json') # downloaded from https://github.com/rreichel3/US-Stock-Symbols.git
    nasdaq_ticker_list = nasdaq_tickers[0].str.replace('.', '-').unique().tolist()
    return nasdaq_ticker_list

In [ ]:
#sp500_tickers = get_sp500_tickers()
nasdaq_ticker_list = get_nasdaq_tickers()
yesterday = (datetime.today()-pd.DateOffset(1)).strftime('%Y-%m-%d')
df = load_data(tickers=nasdaq_ticker_list, end_date=yesterday, years=16)

[******                12%                       ]  460 of 3825 completed

In [5]:
prices = df.stack().rename(columns=str.lower)
prices.index.names = ['date', 'ticker']
prices = prices.drop('close',axis=1)
prices.rename(columns={'adj close':'close'},inplace=True)
#prices = prices.swaplevel()
prices

/tmp/ipykernel_24423/3538683117.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  prices = df.stack().rename(columns=str.lower)


Price                  close       high        low       open       volume
date       ticker                                                         
2009-05-06 AACG     0.453672   6.040000   5.550000   6.030000      32900.0
           AAL      4.713707   5.350000   4.760000   5.240000    6928400.0
           AAME     0.564793   0.640000   0.640000   0.640000        500.0
           AAON     3.511106   4.148148   3.903210   3.909136    1246894.0
           AAPL     3.987403   4.767857   4.650714   4.761786  473538800.0
...                      ...        ...        ...        ...          ...
2025-05-01 ZVRA     7.640000   7.895000   7.190000   7.300000     743500.0
           ZVSA     0.619000   0.630000   0.586000   0.590000     150600.0
           ZYBT    10.430000  10.896000   8.200000   9.000000     424900.0
           ZYME    12.880000  13.080000  12.510000  12.950000     409700.0
           ZYXI     2.010000   2.010000   1.700000   1.750000     647800.0

[7932474 rows x 5 columns]

In [6]:
prices.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 7932474 entries, (Timestamp('2009-05-06 00:00:00'), 'AACG') to (Timestamp('2025-05-01 00:00:00'), 'ZYXI')
Data columns (total 5 columns):
 #   Column  Dtype  
---  ------  -----  
 0   close   float64
 1   high    float64
 2   low     float64
 3   open    float64
 4   volume  float64
dtypes: float64(5)
memory usage: 333.2+ MB


In [7]:
with pd.HDFStore(DATA_STORE, mode='w') as store:
    store.put('nasdaq/price', prices)

### Get Metadata

In [8]:
metadata = pd.read_json('../data/nasdaq_full_tickers.json').rename(columns={'symbol':'ticker'})
metadata['ticker'] = metadata['ticker'].str.replace('.', '-').unique()
metadata = metadata.set_index('ticker')
metadata.columns = metadata.columns.str.lower()

In [9]:
metadata

,name,lastsale,netchange,pctchange,volume,marketcap,country,ipoyear,industry,sector,url
ticker,,,,,,,,,,,
AACB,Artius II Acquisition Inc. Class A Ordinary Sh...,$9.92,-0.0100,-0.101%,863065,0.00,United States,2025,,,/market-activity/stocks/aacb
AACBR,Artius II Acquisition Inc. Rights,$0.20,-0.0063,-3.054%,278070,0.00,United States,2025,,,/market-activity/stocks/aacbr
AACBU,Artius II Acquisition Inc. Units,$10.1099,0.0000,0.00%,2518,0.00,United States,2025,Blank Checks,Finance,/market-activity/stocks/aacbu
AACG,ATA Creativity Global American Depositary Shares,$0.905,-0.0550,-5.729%,22291,28879805.00,China,2008,Other Consumer Services,Real Estate,/market-activity/stocks/aacg
AAL,American Airlines Group Inc. Common Stock,$9.75,0.1400,1.457%,65137090,6411361925.00,United States,,Air Freight/Delivery Services,Consumer Discretionary,/market-activity/stocks/aal
...,...,...,...,...,...,...,...,...,...,...,...
ZVRA,Zevra Therapeutics Inc. Common Stock,$7.20,-0.0400,-0.552%,381265,389634257.00,United States,,Biotechnology: Pharmaceutical Preparations,Health Care,/market-activity/stocks/zvra
ZVSA,ZyVersa Therapeutics Inc. Common Stock,$0.701,-0.0141,-1.972%,71919,1800302.00,United States,2022,Biotechnology: Pharmaceutical Preparations,Health Care,/market-activity/stocks/zvsa
ZYBT,Zhengye Biotechnology Holding Limited Ordinary...,$10.44,-0.1100,-1.043%,599382,492416965.00,China,2025,Biotechnology: Pharmaceutical Preparations,Health Care,/market-activity/stocks/zybt


In [10]:
# def get_metadata(tickers):
#     metadata = []

#     for ticker in tickers:
#         try:
#             t = yf.Ticker(ticker)
#             info = t.info

#             data = {
#                 'ticker': ticker,
#                 #'lastsale': info.get('regularMarketPrice'),
#                 'marketCap': info.get('marketCap'),
#                 #'ipoyear': info.get('ipoDate'),  # Might need parsing
#                 'sector': info.get('sector'),
#                 'industry': info.get('industry')
#             }
#             metadata.append(data)
#         except Exception as e:
#             print(f"Failed to get info for {ticker}: {e}")
#             continue

#     meta_df = pd.DataFrame(metadata)
#     return meta_df

# metadata = get_metadata(nasdaq_ticker_list)
# metadata = metadata.set_index('ticker')
# metadata.sort_index()

In [11]:
# idx = pd.IndexSlice
# shared = (prices.index.get_level_values('ticker').unique()
#           .intersection(metadata.index))
# metadata = metadata.loc[shared, :]
# prices = prices.loc[idx[shared, :], :]

In [ ]:
#with pd.HDFStore(DATA_STORE, mode='w') as store:
with pd.HDFStore(DATA_STORE) as store:
    store.put('nasdaq/metadata', metadata)

: 